In [3]:
import sys
from pathlib import Path

# If notebook is inside the notebooks/ folder, go up one level to project root
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("app exists:", (project_root / "app").exists())

Project root: c:\Users\teren\Documents\Machine Learning\can-ids-binary-detector
app exists: True


In [4]:
from app.predict import load_artifacts, preprocess_input

In [5]:
model, scaler, threshold = load_artifacts()

print("Artifacts loaded successfully.")
print("Threshold:", threshold)

Artifacts loaded successfully.
Threshold: 0.9000000000000002


In [6]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sample_file = BASE_DIR / "sample_data" / "sample_input.csv"

df = pd.read_csv(sample_file)
df.head()

,dlc,can_id_int,byte_0_int,byte_1_int,byte_2_int,byte_3_int,byte_4_int,byte_5_int,byte_6_int,byte_7_int,inter_arrival,payload_sum,nonzero_bytes,payload_unique_values
0,8,123,10,20,0,0,5,1,0,255,0.0020,291,5,6
1,8,250,0,0,0,0,0,0,0,0,0.0100,0,0,1
2,8,512,34,12,9,100,2,3,4,5,0.0010,169,8,8
3,8,700,255,255,255,255,255,255,255,255,0.0005,2040,8,1
4,4,321,1,2,3,4,0,0,0,0,0.0200,10,4,4


In [7]:
X = preprocess_input(df)
print("Processed shape:", X.shape)
X.head()

Processed shape: (5, 14)


,dlc,can_id_int,byte_0_int,byte_1_int,byte_2_int,byte_3_int,byte_4_int,byte_5_int,byte_6_int,byte_7_int,inter_arrival,payload_sum,nonzero_bytes,payload_unique_values
0,8,123,10,20,0,0,5,1,0,255,0.0020,291,5,6
1,8,250,0,0,0,0,0,0,0,0,0.0100,0,0,1
2,8,512,34,12,9,100,2,3,4,5,0.0010,169,8,8
3,8,700,255,255,255,255,255,255,255,255,0.0005,2040,8,1
4,4,321,1,2,3,4,0,0,0,0,0.0200,10,4,4


In [8]:
X_scaled = scaler.transform(X)
print("Scaled feature matrix shape:", X_scaled.shape)

Scaled feature matrix shape: (5, 14)


In [9]:
probs = model.predict(X_scaled, verbose=0).ravel()
preds = (probs >= threshold).astype(int)

results = df.copy()
results["predicted_probability"] = probs
results["predicted_class"] = preds

results

,dlc,can_id_int,byte_0_int,byte_1_int,byte_2_int,byte_3_int,byte_4_int,byte_5_int,byte_6_int,byte_7_int,inter_arrival,payload_sum,nonzero_bytes,payload_unique_values,predicted_probability,predicted_class
0,8,123,10,20,0,0,5,1,0,255,0.0020,291,5,6,6.979281e-01,0
1,8,250,0,0,0,0,0,0,0,0,0.0100,0,0,1,9.931097e-01,1
2,8,512,34,12,9,100,2,3,4,5,0.0010,169,8,8,1.940515e-05,0
3,8,700,255,255,255,255,255,255,255,255,0.0005,2040,8,1,9.999999e-01,1
4,4,321,1,2,3,4,0,0,0,0,0.0200,10,4,4,5.324244e-17,0


In [10]:
output_file = BASE_DIR / "sample_data" / "predictions_output.csv"
results.to_csv(output_file, index=False)
print(f"Saved predictions to: {output_file}")

Saved predictions to: c:\Users\teren\Documents\Machine Learning\can-ids-binary-detector\sample_data\predictions_output.csv
